In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

K = np.array([
    3.5e0,   5.0e1,   1.0e4,   5.0e-2,  5.0e-2,    
    2.0e4,   5.0e-3,  5.0e4,   2.0e4,   1.0e5,    
    5.0e-1,  2.0e4,   3.0e-1,  1.0e4,   1.0e6,   
    1.0e-2,  2.0e0,   5.0e5,   2.0e6,   3.0e3,    
    3.0e0,   8.0e0,   5.0e0,   3.0e3,   5.0e0     
])


Y0 = np.array([
    0,      1e3,    0,      3e2,    0,    
    5e1,    8e2,    1.5e3,  3e1,    0,    
    0,      0,      0,      0,      0, 
    1e2,    5e1,    1,      0,      0  
])

def reaction_rates(y, k):
    """计算25个反应速率 r1-r25"""
    r = np.zeros(25)
    r[0]  = k[0] * y[0]               # r1 = k1*y1
    r[1]  = k[1] * y[1] * y[3]        # r2 = k2*y2*y4
    r[2]  = k[2] * y[4] * y[1]        # r3 = k3*y5*y2
    r[3]  = k[3] * y[6]               # r4 = k4*y7
    r[4]  = k[4] * y[6]               # r5 = k5*y7
    r[5]  = k[5] * y[6] * y[5]        # r6 = k6*y7*y6
    r[6]  = k[6] * y[8]               # r7 = k7*y9
    r[7]  = k[7] * y[8] * y[5]        # r8 = k8*y9*y6
    r[8]  = k[8] * y[10] * y[1]       # r9 = k9*y11*y2
    r[9]  = k[9] * y[10] * y[0]       # r10 = k10*y11*y1
    r[10] = k[10] * y[12]             # r11 = k11*y13
    r[11] = k[11] * y[9] * y[1]       # r12 = k12*y10*y2
    r[12] = k[12] * y[13]             # r13 = k13*y14
    r[13] = k[13] * y[0] * y[5]       # r14 = k14*y1*y6
    r[14] = k[14] * y[2]              # r15 = k15*y3
    r[15] = k[15] * y[3]              # r16 = k16*y4
    r[16] = k[16] * y[3]              # r17 = k17*y4
    r[17] = k[17] * y[15]             # r18 = k18*y16
    r[18] = k[18] * y[15]             # r19 = k19*y16
    r[19] = k[19] * y[16] * y[5]      # r20 = k20*y17*y6
    r[20] = k[20] * y[18]             # r21 = k21*y19
    r[21] = k[21] * y[18]             # r22 = k22*y19
    r[22] = k[22] * y[0] * y[3]       # r23 = k23*y1*y4
    r[23] = k[23] * y[18] * y[0]      # r24 = k24*y19*y1
    r[24] = k[24] * y[19]             # r25 = k25*y20
    return r

def pollu_ode(t, y, k=K):
    """POLLU ODE系统: dy/dt = N * r(y; k)"""
    r = reaction_rates(y, k)
    dydt = np.zeros(20)
    
    dydt[0]  = -r[0] - r[9] - r[13] - r[22] - r[23] + r[1] + r[2] + r[8] + r[10] + r[11] + r[21] + r[24]
    dydt[1]  = -r[1] - r[2] - r[8] - r[11] + r[0] + r[20]
    dydt[2]  = -r[14] + r[0] + r[16] + r[18] + r[21]
    dydt[3]  = -r[1] - r[15] - r[16] - r[22] + r[14]
    dydt[4]  = -r[2] + 2*r[3] + r[5] + r[6] + r[12] + r[19]
    dydt[5]  = -r[5] - r[7] - r[13] - r[19] + r[2] + 2*r[17]
    dydt[6]  = -r[3] - r[4] - r[5] + r[12]
    dydt[7]  = r[3] + r[4] + r[5] + r[6]
    dydt[8]  = -r[6] - r[7]
    dydt[9]  = -r[11] + r[6] + r[8]
    dydt[10] = -r[8] - r[9] + r[7] + r[10]
    dydt[11] = r[8]
    dydt[12] = -r[10] + r[9]
    dydt[13] = -r[12] + r[11]
    dydt[14] = r[13]
    dydt[15] = -r[17] - r[18] + r[15]
    dydt[16] = -r[19]
    dydt[17] = r[19]
    dydt[18] = -r[20] - r[21] - r[23] + r[22] + r[24]
    dydt[19] = -r[24] + r[23]
    
    return dydt

def generate_data(t_span=(1e-12, 1e4), n_points=1000):
    t_eval = np.logspace(np.log10(t_span[0]), np.log10(t_span[1]), n_points)
    sol = solve_ivp(pollu_ode, t_span, Y0, method='BDF', t_eval=t_eval, atol=1e-10, rtol=1e-8)
    if not sol.success:
        raise RuntimeError(f"ODE求解失败: {sol.message}")
    return sol.t, sol.y.T

def create_time_features(t):
    t = np.log10(t + 1e-12)
    return t.reshape(-1, 1)


class ResidualBlock(nn.Module):
    """残差块"""
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.act = nn.ReLU()
        self.norm = nn.LayerNorm(dim)
    
    def forward(self, x):
        out = self.act(self.fc1(x))
        out = self.fc2(out)
        return self.act(self.norm(out + x))


class ResidualMLP(nn.Module):
    def __init__(self, input_dim, output_dim=20, hidden_dim=128, num_layers=3):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )
        self.blocks = nn.ModuleList([ResidualBlock(hidden_dim) for _ in range(num_layers)])
        self.output_proj = nn.Linear(hidden_dim, output_dim)
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        h = self.input_proj(x)
        for block in self.blocks:
            h = block(h)
        return self.output_proj(h)

class POLLUDataset(Dataset):
    def __init__(self, t, y):
        self.x = create_time_features(t).astype(np.float32)
        self.y = y.astype(np.float32)
        
        # 归一化
        self.y_mean = self.y.mean(axis=0, keepdims=True)
        self.y_std = self.y.std(axis=0, keepdims=True) + 1e-8
        self.y_norm = (self.y - self.y_mean) / self.y_std
        
        self.x_mean = self.x.mean(axis=0, keepdims=True)
        self.x_std = self.x.std(axis=0, keepdims=True) + 1e-8
        self.x_norm = (self.x - self.x_mean) / self.x_std
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return torch.from_numpy(self.x_norm[idx]), torch.from_numpy(self.y_norm[idx])

def train(model, dataset, epochs=2500, batch_size=64, lr=1e-3, device='cpu'):
    model = model.to(device)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    criterion = nn.MSELoss()
    
    losses = []
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model(x_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        scheduler.step()
        avg_loss = epoch_loss / len(dataloader)
        losses.append(avg_loss)
        
        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}")
    
    return losses

def plot_results(t, y_true, y_pred, losses, save_prefix='pollu'):
    plt.figure(figsize=(10, 6))
    plt.semilogy(losses)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.grid(True, alpha=0.3)
    plt.savefig(f'{save_prefix}_loss.png', dpi=150, bbox_inches='tight')
    plt.close()

    fig, axes = plt.subplots(4, 5, figsize=(20, 16))
    axes = axes.flatten()
    for i in range(20):
        ax = axes[i]
        mae = np.mean(np.abs(y_true[:, i] - y_pred[:, i]))
        ax.semilogx(t, y_true[:, i], 'b-', label='Analytical', linewidth=1.5)
        ax.semilogx(t, y_pred[:, i], 'r--', label='NN', linewidth=1.5)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Conc. (M)')
        ax.set_title(f'$y_{{{i+1}}}$ MAE={mae:.2e}')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
    plt.suptitle('POLLU problem', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    # 整体误差
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    mae_per_t = np.mean(np.abs(y_true - y_pred), axis=1)
    rmse_per_t = np.sqrt(np.mean((y_true - y_pred)**2, axis=1))
    
    axes[0].loglog(t, mae_per_t, 'b-', label='MAE')
    axes[0].loglog(t, rmse_per_t, 'r--', label='RMSE')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Error')
    axes[0].set_title('Overall Error vs Time')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    mae_per_species = np.mean(np.abs(y_true - y_pred), axis=0)
    axes[1].bar(range(1, 21), mae_per_species)
    axes[1].set_xlabel('Species')
    axes[1].set_ylabel('MAE')
    axes[1].set_title('MAE per Species')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_error.png', dpi=150, bbox_inches='tight')
    plt.close()



def main():
    t, y = generate_data(n_points=1000)
    dataset = POLLUDataset(t, y)
    input_dim = dataset.x.shape[1]
    model = ResidualMLP(input_dim=input_dim, output_dim=20, hidden_dim=64, num_layers=3)
    n_params = sum(p.numel() for p in model.parameters())
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    losses = train(model, dataset, epochs=10000, batch_size=64, lr=1e-3, device=device)
    model.eval()
    with torch.no_grad():
        x_tensor = torch.from_numpy(dataset.x_norm).to(device)
        y_pred_norm = model(x_tensor).cpu().numpy()
    
    y_pred = y_pred_norm * dataset.y_std + dataset.y_mean

    mae = np.mean(np.abs(y - y_pred))
    rmse = np.sqrt(np.mean((y - y_pred)**2))
    print(f"    MAE: {mae:.4e}")
    print(f"    RMSE: {rmse:.4e}")
    
    return model, t, y, y_pred, losses


if __name__ == "__main__":
    model, t, y, y_pred, losses = main()

Epoch 100/10000, Loss: 0.001553
Epoch 200/10000, Loss: 0.000359
Epoch 300/10000, Loss: 0.000186
Epoch 400/10000, Loss: 0.000178
Epoch 500/10000, Loss: 0.000140
Epoch 600/10000, Loss: 0.000065
Epoch 700/10000, Loss: 0.000195
Epoch 800/10000, Loss: 0.000050
Epoch 900/10000, Loss: 0.000108
Epoch 1000/10000, Loss: 0.000021
Epoch 1100/10000, Loss: 0.000038
Epoch 1200/10000, Loss: 0.000499
Epoch 1300/10000, Loss: 0.000012
Epoch 1400/10000, Loss: 0.000014
Epoch 1500/10000, Loss: 0.000834
Epoch 1600/10000, Loss: 0.000008
Epoch 1700/10000, Loss: 0.000031
Epoch 1800/10000, Loss: 0.000008
Epoch 1900/10000, Loss: 0.000010
Epoch 2000/10000, Loss: 0.000066
Epoch 2100/10000, Loss: 0.000007
Epoch 2200/10000, Loss: 0.000268
Epoch 2300/10000, Loss: 0.000008
Epoch 2400/10000, Loss: 0.000023
Epoch 2500/10000, Loss: 0.000288
Epoch 2600/10000, Loss: 0.000016
Epoch 2700/10000, Loss: 0.000096
Epoch 2800/10000, Loss: 0.000027
Epoch 2900/10000, Loss: 0.000192
Epoch 3000/10000, Loss: 0.000007
Epoch 3100/10000, L